# Consolidated Dataset Preprocessing & Master Consolidation

This notebook consolidates GolfDB and UCF non-golf single video CSV files, performs sliding window feature engineering, maps milestone event labels, and outputs a unified consolidated master dataset `data/processed/master_dataset.csv` for training models.

In [ ]:
# Import required libraries
%load_ext autoreload
%autoreload 2

import os
import sys
import glob
import ast
import pandas as pd
import numpy as np

# Set project root path relative to this notebook's directory
PROJECT_ROOT = os.path.abspath("../")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.feature_engineer import engineer_sliding_window, HIGH_MOVEMENT_JOINTS

## 1. Directory and File Paths Setup

In [ ]:
GOLF_FEATURES_DIR = os.path.join(PROJECT_ROOT, "data/processed/single_video_features/golf")
UCF_FEATURES_DIR = os.path.join(PROJECT_ROOT, "data/processed/single_video_features/non_golf")
GOLFDB_METADATA_PATH = os.path.join(PROJECT_ROOT, "data/GolfDB.csv")
MASTER_CSV_PATH = os.path.join(PROJECT_ROOT, "data/processed/master_dataset.csv")

## 2. Load GolfDB Metadata & Event Mapping

In [ ]:
print("Loading GolfDB metadata...")
df_meta = pd.read_csv(GOLFDB_METADATA_PATH)
meta_dict = {}
for _, row in df_meta.iterrows():
    vid_id = int(row['id'])
    events = ast.literal_eval(row['events'])
    start_frame = events[0]
    # Milestones relative to the cropped video start
    milestones = [ev - start_frame for ev in events[1:9]]
    meta_dict[vid_id] = {
        'milestones': milestones,
        'player': row.get('player', 'unknown'),
        'sex': row.get('sex', 'unknown'),
        'club': row.get('club', 'unknown'),
        'view': row.get('view', 'unknown'),
        'slow': int(row.get('slow', 0))
    }
print(f"Loaded metadata mapping for {len(meta_dict)} golf videos.")

## 3. Consolidate and Feature Engineer Datasets

In [ ]:
golf_files = sorted(glob.glob(os.path.join(GOLF_FEATURES_DIR, "*.csv")), key=lambda x: int(os.path.basename(x).split('.')[0]))
ucf_files = sorted(glob.glob(os.path.join(UCF_FEATURES_DIR, "*.csv")))

print(f"Found {len(golf_files)} golf files and {len(ucf_files)} non-golf files.")

merged_dfs = []

# Process GolfDB swings
print("Processing GolfDB swings...")
for i, f in enumerate(golf_files):
    vid_id = int(os.path.basename(f).split('.')[0])
    df = pd.read_csv(f)
    
    # Apply sliding window feature engineering if not present
    if "norm_left_wrist_x_t-5" not in df.columns:
        df = engineer_sliding_window(df, joints=HIGH_MOVEMENT_JOINTS)
        
    df['is_golf'] = 1
    df['label'] = 0
    
    # Map milestone event labels 1-8
    if vid_id in meta_dict:
        meta = meta_dict[vid_id]
        for milestone_idx, frame_idx in enumerate(meta['milestones']):
            label_val = milestone_idx + 1
            mask = df['frame_index'] == frame_idx
            df.loc[mask, 'label'] = label_val
            
        for k, v in meta.items():
            if k != 'milestones':
                df[k] = v
    else:
        df['player'] = 'unknown'
        df['sex'] = 'unknown'
        df['club'] = 'unknown'
        df['view'] = 'unknown'
        df['slow'] = 0
        
    # Drop unused coordinates and scale parameters to optimize size
    cols_to_keep = [c for c in df.columns if not (c.startswith("raw_") or c.startswith("smooth_") or c in [
        "mid_hip_x", "mid_hip_y", "mid_shoulder_x", "mid_shoulder_y", 
        "torso_length", "torso_scale", "width", "height", "fps"
    ])]
    df = df[cols_to_keep]
    merged_dfs.append(df)
    
    if (i + 1) % 300 == 0 or (i + 1) == len(golf_files):
        print(f"  Processed {i + 1}/{len(golf_files)} golf videos.")

# Process UCF non-golf videos
print("\nProcessing UCF non-golf videos...")
for i, f in enumerate(ucf_files):
    df = pd.read_csv(f)
    
    # Apply sliding window feature engineering if not present
    if "norm_left_wrist_x_t-5" not in df.columns:
        df = engineer_sliding_window(df, joints=HIGH_MOVEMENT_JOINTS)
        
    df['is_golf'] = 0
    df['label'] = 0
    
    # Fill non-golf metadata defaults
    df['player'] = 'none'
    df['sex'] = 'none'
    df['club'] = 'none'
    df['view'] = 'none'
    df['slow'] = 0
    
    cols_to_keep = [c for c in df.columns if not (c.startswith("raw_") or c.startswith("smooth_") or c in [
        "mid_hip_x", "mid_hip_y", "mid_shoulder_x", "mid_shoulder_y", 
        "torso_length", "torso_scale", "width", "height", "fps"
    ])]
    df = df[cols_to_keep]
    merged_dfs.append(df)
    
    if (i + 1) % 300 == 0 or (i + 1) == len(ucf_files):
        print(f"  Processed {i + 1}/{len(ucf_files)} non-golf videos.")

## 4. Concatenate and Save Consolidated Master Dataset

In [ ]:
print("Concatenating all dataframes...")
df_master = pd.concat(merged_dfs, ignore_index=True)
print(f"Final consolidated master dataset shape: {df_master.shape}")

print(f"Saving unified master dataset to {MASTER_CSV_PATH}...")
df_master.to_csv(MASTER_CSV_PATH, index=False)
print("Successfully saved unified master dataset!")